# PFE - Benchmark des modeles LLM (Ollama)

Objectif : comparer trois modeles Ollama sur des questions de qualite des donnees SQL Server.
Le notebook utilise le prompt systeme de base pour tous les modeles afin de comparer les sorties.

In [60]:
# 1. Initialisation de l'environnement Django et import Ollama
import os
import sys
import json
import time
from typing import Optional

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mon_projet.settings")
# Jupyter peut executer dans un contexte async : autoriser l'ORM en mode sync si besoin.
os.environ.setdefault("DJANGO_ALLOW_ASYNC_UNSAFE", "true")

import django
django.setup()

from engine.ollama_client import chat_with_model, OllamaError
from engine.views import _build_analyses_summary
from engine.models import LLMResponseLog

print("Django initialise. Racine projet :", PROJECT_ROOT)

Django initialise. Racine projet : c:\Users\wessh\PFE


In [61]:
# 2. Chargement du resume des analyses (peut prendre quelques secondes)
start = time.perf_counter()
try:
    analyses_summary = _build_analyses_summary(max_bases=30)
    duree = time.perf_counter() - start
    print(f"Resume charge : {len(analyses_summary)} bases en {duree:.2f} s")
except Exception as exc:
    print("Erreur lors du chargement des analyses :", exc)
    analyses_summary = []
    print("Le benchmark pourra s'executer sans contexte reel.")

Resume charge : 7 bases en 0.10 s


In [62]:
# 3. Prompt systeme de base (a adapter si besoin)
SYSTEM_PROMPT_BASELINE = (
    "Rôle : Tu es un assistant expert en gouvernance et en qualité des données "
    "sur des bases SQL Server dans un contexte de projet de fin d'études. "
    "Tu connais les bonnes pratiques de modélisation, de contraintes et d'indexation.\n\n"
    "Tâche : À partir des métriques fournies pour chaque base (score global, nombre de tables, "
    "tables sans clé primaire, sans index, jamais utilisées, nombre de procédures, vues et FK), "
    "tu réponds à toute question liée à la qualité des données, à la performance, "
    "à la fiabilité ou à la gouvernance.\n\n"
    "Structure de sortie :\n"
    "1) Une courte introduction qui rappelle le contexte de la réponse.\n"
    "2) 3 à 5 points clés structurés (puces) qui justifient ton analyse.\n"
    "3) Une conclusion brève avec une recommandation claire.\n\n"
    "Contraintes : Tu réponds toujours en français, avec un ton professionnel, clair et synthétique. "
    "Tu t'appuies uniquement sur les données fournies dans le contexte et tu évites de spéculer. "
    "Si une information manque, mentionne la limite sans inventer. "
    "Chaque point clé cite au moins une métrique (score, tables, PK, index, tables jamais utilisées, procédures, vues, FK). "
    "Indique un niveau de certitude (forte, moyenne, faible). "
    "Longueur maximale : 150 mots. "
    "Utilise les termes normalisés : PK, index, FK, tables jamais utilisées."
 )

In [63]:
# 3bis. Prompt avance : few-shot + raisonnement interne (non affiche)
SYSTEM_PROMPT_ADVANCED = (
    "Rôle : Tu es un assistant expert en gouvernance et en qualité des données "
    "sur des bases SQL Server dans un contexte de projet de fin d'études. "
    "Tu connais les bonnes pratiques de modélisation, de contraintes et d'indexation.\n\n"
    "Tâche : À partir des métriques fournies pour chaque base (score global, nombre de tables, "
    "tables sans clé primaire, sans index, jamais utilisées, nombre de procédures, vues et FK), "
    "tu réponds à toute question liée à la qualité des données, à la performance, "
    "à la fiabilité ou à la gouvernance.\n\n"
    "Processus de raisonnement interne (non affiché) :\n"
    "- Identifie d'abord les bases les plus pertinentes pour la question.\n"
    "- Analyse les indicateurs techniques (PK, index, tables jamais utilisées, FK, vues, procédures).\n"
    "- Formule mentalement un raisonnement étape par étape, puis ne fournis au final "
    "qu'un résumé structuré et compréhensible, sans détailler toutes les étapes.\n\n"
    "Structure de sortie :\n"
    "1) Une courte introduction qui rappelle le contexte de la réponse.\n"
    "2) 3 à 5 points clés structurés (puces) qui justifient ton analyse.\n"
    "3) Une conclusion brève avec une recommandation claire.\n\n"
    "Contraintes : Tu réponds toujours en français, avec un ton professionnel, clair et synthétique. "
    "Tu t'appuies uniquement sur les données fournies dans le contexte et tu évites de spéculer. "
    "Si une information manque, mentionne la limite sans inventer. "
    "Chaque point clé cite au moins une métrique (score, tables, PK, index, tables jamais utilisées, procédures, vues, FK). "
    "Indique un niveau de certitude (forte, moyenne, faible). "
    "Longueur maximale : 150 mots. "
    "Utilise les termes normalisés : PK, index, FK, tables jamais utilisées."
 )

FEW_SHOT_EXAMPLES = [
    {
        "question": "Quelle est la meilleure base analysée et pourquoi ?",
        "answer": (
            "La meilleure base est celle qui présente le score global le plus élevé, "
            "avec peu de tables sans PK ni index, et peu de tables jamais utilisées. "
            "La réponse met en avant la bonne structuration, l'utilisation régulière des tables "
            "et la présence d'index adaptés."
        ),
    },
    {
        "question": "Quelles sont les principales faiblesses de la base la moins bien notée ?",
        "answer": (
            "La base la moins bien notée cumule plusieurs points faibles : nombreuses tables sans PK, "
            "absence d'index sur des tables volumineuses, et trop de tables jamais utilisées. "
            "La réponse insiste sur l'impact de ces problèmes sur la qualité, la performance "
            "et la maintenabilité des données."
        ),
    },
    {
        "question": "Donne trois recommandations prioritaires pour améliorer la qualité des données.",
        "answer": (
            "Trois actions prioritaires sont proposées, par exemple : (1) ajouter des PK "
            "sur les tables critiques, (2) créer des index sur les colonnes fréquemment filtrées, "
            "(3) archiver ou nettoyer les tables jamais utilisées. Chaque recommandation est justifiée "
            "par son impact sur la fiabilité et les performances."
        ),
    },
    {
        "question": "Quelle base présente le plus de risques opérationnels et pourquoi ?",
        "answer": (
            "La base la plus risquée est celle qui cumule un faible score global, un nombre élevé de "
            "tables sans PK et sans index, ainsi qu'une proportion importante de tables jamais utilisées. "
            "Ces signaux indiquent des risques de cohérence, de performance et de maîtrise des données."
        ),
    },
    {
        "question": "Sur quelle base faut-il prioriser un plan d'indexation ?",
        "answer": (
            "La priorité va à la base qui présente le plus grand nombre de tables sans index, "
            "surtout si le volume de tables est élevé. Cette situation pénalise les performances "
            "et augmente les temps de réponse sur les requêtes courantes."
        ),
    },
 ]

In [64]:
# 4. Construction du message utilisateur
def build_user_content(question: str, analyses_summary: list[dict]) -> str:
    analyses_json = json.dumps(analyses_summary, ensure_ascii=False, indent=2)
    parts: list[str] = []
    parts.append("Resume des analyses de bases de donnees (format JSON, une entree par base) :\n\n")
    parts.append(analyses_json + "\n\n")
    parts.append("Question de l'utilisateur :\n")
    parts.append(question + "\n\n")
    parts.append("Reponds uniquement en francais, de facon professionnelle et synthetique. "
                 "Appuie-toi sur les metriques fournies (score, nombre de tables, tables sans PK/index, "
                 "tables jamais utilisees, procedures, vues, FK). "
                 "Mets en avant les elements les plus importants pour la decision et termine par une recommandation claire.")
    return "".join(parts)

In [65]:
# 5. Fonctions utilitaires
from django.core.exceptions import SynchronousOnlyOperation

def run_single_query(
    model_name: str,
    question: str,
    analyses_summary: Optional[list[dict]] = None,
    temperature: float = 0.2,
    log_to_db: bool = False,
 ) -> dict:
    if analyses_summary is None:
        analyses_summary = _build_analyses_summary(max_bases=30)

    user_content = build_user_content(question, analyses_summary)

    print(f"=== Modele : {model_name} ===")
    start = time.perf_counter()
    try:
        answer, duration_s = chat_with_model(
            model=model_name,
            system_prompt=SYSTEM_PROMPT_BASELINE,
            user_content=user_content,
            temperature=temperature,
        )
        duration = duration_s
        print(f"Temps de reponse : {duration:.2f} s\n")
        print(answer)
        result = {
            "model": model_name,
            "question": question,
            "answer": answer,
            "duration_s": duration,
            "error": None,
        }
        if log_to_db:
            try:
                LLMResponseLog.objects.create(
                    model_name=model_name,
                    question=question,
                    answer=answer,
                    latency_ms=duration * 1000.0,
                )
            except SynchronousOnlyOperation:
                print(
                    "Logging base desactive (contexte async). "
                    "Mets DJANGO_ALLOW_ASYNC_UNSAFE=true ou passe log_to_db=False."
                )
    except OllamaError as exc:
        print(f"Erreur Ollama : {exc}")
        result = {
            "model": model_name,
            "question": question,
            "answer": "",
            "duration_s": None,
            "error": str(exc),
        }
    return result

def benchmark_models(
    questions: list[str],
    models: list[str],
    analyses_summary: list[dict],
    temperature: float = 0.2,
    log_to_db: bool = False,
 ) -> list[dict]:
    results: list[dict] = []
    for q in questions:
        print("\n######## Question :", q, "\n")
        for m in models:
            res = run_single_query(
                model_name=m,
                question=q,
                analyses_summary=analyses_summary,
                temperature=temperature,
                log_to_db=log_to_db,
            )
            results.append(res)
    return results

In [66]:
# 6. Questions et modeles a benchmarker
TEST_QUESTIONS: list[str] = [
    "Quelle est la meilleure base analysee et pourquoi ?",
    "Quelles sont les principales faiblesses de la base la moins bien notee ?",
    "Donne trois recommandations prioritaires pour ameliorer la qualite globale des donnees.",
]

MODELS: list[str] = [
    "llama3:8b",
    "qwen2.5:7b",
    "deepseek-r1:7b",
]

In [67]:
# 7. Test manuel sur un modele
question = TEST_QUESTIONS[0]
model_name = MODELS[0]

result_example = run_single_query(
    model_name=model_name,
    question=question,
    analyses_summary=analyses_summary,
    temperature=0.2,
    log_to_db=False,
 )

result_example

=== Modele : llama3:8b ===
Temps de reponse : 56.98 s

Introduction :

La question porte sur la meilleure base analysee parmi celles fournies. Pour répondre à cette question, nous allons examiner les métriques fournies pour chaque base.

Points clés :

* La base "pfe_db" (serveur_id 7 et 8) présente un score global élevé (96,0), ce qui indique une bonne qualité des données.
* Cette base a également un nombre raisonnable de tables (15) et un faible nombre de tables sans clé primaire (1).
* Cependant, il est important de noter que cette base a un nombre relativement élevé de tables jamais utilisées (7), ce qui peut indiquer une certaine inertie dans l'utilisation des données.
* Les autres bases ont des scores globaux plus faibles et des métriques moins favorables.

Conclusion :

En résumé, la meilleure base analysee est "pfe_db" (serveur_id 7 et 8) en raison de son score global élevé et de ses métriques relativement favorables. Cependant, il est important de prendre en compte le fait que

{'model': 'llama3:8b',
 'question': 'Quelle est la meilleure base analysee et pourquoi ?',
 'answer': 'Introduction :\n\nLa question porte sur la meilleure base analysee parmi celles fournies. Pour répondre à cette question, nous allons examiner les métriques fournies pour chaque base.\n\nPoints clés :\n\n* La base "pfe_db" (serveur_id 7 et 8) présente un score global élevé (96,0), ce qui indique une bonne qualité des données.\n* Cette base a également un nombre raisonnable de tables (15) et un faible nombre de tables sans clé primaire (1).\n* Cependant, il est important de noter que cette base a un nombre relativement élevé de tables jamais utilisées (7), ce qui peut indiquer une certaine inertie dans l\'utilisation des données.\n* Les autres bases ont des scores globaux plus faibles et des métriques moins favorables.\n\nConclusion :\n\nEn résumé, la meilleure base analysee est "pfe_db" (serveur_id 7 et 8) en raison de son score global élevé et de ses métriques relativement favorables

In [68]:
# 8. Benchmark complet
LOG_TO_DB = False

all_results = benchmark_models(
    questions=TEST_QUESTIONS,
    models=MODELS,
    analyses_summary=analyses_summary,
    temperature=0.2,
    log_to_db=LOG_TO_DB,
 )

all_results[:3]


######## Question : Quelle est la meilleure base analysee et pourquoi ? 

=== Modele : llama3:8b ===
Temps de reponse : 51.35 s

Introduction :

La question se pose quant à la meilleure base analysee parmi celles fournies. Pour répondre à cette interrogation, nous allons nous appuyer sur les métriques fournies pour chaque base.

Points clés :

• La base "pfe_db" (serveur_id 7 et 8) présente un score global élevé (96,0), ce qui indique une bonne qualité des données. Le nombre de tables est raisonnable (15), et le nombre de procédures et de vues est nul, ce qui signifie que la base ne contient pas d'éléments inutiles.

• La base "Test1" (serveur_id 7) présente un score global plus faible (74,0). Le nombre de tables est relativement élevé (7), mais le nombre de procédures et de vues est raisonnable (3).

• La base "Test4" (serveur_id 8) présente un score global modéré (79,0). Le nombre de tables est faible (3), ce qui peut indiquer une certaine spécialisation.

Conclusion :

En résumé, l

[{'model': 'llama3:8b',
  'question': 'Quelle est la meilleure base analysee et pourquoi ?',
  'answer': 'Introduction :\n\nLa question se pose quant à la meilleure base analysee parmi celles fournies. Pour répondre à cette interrogation, nous allons nous appuyer sur les métriques fournies pour chaque base.\n\nPoints clés :\n\n• La base "pfe_db" (serveur_id 7 et 8) présente un score global élevé (96,0), ce qui indique une bonne qualité des données. Le nombre de tables est raisonnable (15), et le nombre de procédures et de vues est nul, ce qui signifie que la base ne contient pas d\'éléments inutiles.\n\n• La base "Test1" (serveur_id 7) présente un score global plus faible (74,0). Le nombre de tables est relativement élevé (7), mais le nombre de procédures et de vues est raisonnable (3).\n\n• La base "Test4" (serveur_id 8) présente un score global modéré (79,0). Le nombre de tables est faible (3), ce qui peut indiquer une certaine spécialisation.\n\nConclusion :\n\nEn résumé, la meilleu

In [69]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [70]:
# 9. Mise en forme des resultats et export CSV
try:
    import pandas as pd
except ImportError:
    raise ImportError(
        "Le module pandas n'est pas installe. Installe-le avec `pip install pandas` dans ton environnement."
    )

df_results = pd.DataFrame(all_results)
df_results[["model", "question", "duration_s", "error"]]

df_results.to_csv("llm_benchmark_results.csv", index=False, encoding="utf-8")

In [71]:
# 10. Comparaison automatique des modeles (score simple)
import re

def score_answer(answer: str) -> dict:
    text = answer or ""
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    lowered = text.lower()

    # Heuristiques simples de qualite
    has_intro = any("contexte" in line.lower() for line in lines[:3])
    has_bullets = any(line.startswith("-") or line.startswith("*") for line in lines)
    has_conclusion = any("recommandation" in line.lower() or "conclusion" in line.lower() for line in lines[-3:])
    mentions_metrics = any(
        term in lowered
        for term in ["score", "pk", "index", "fk", "tables jamais utilisées", "tables jamais utilisees", "procédures", "procedures", "vues"]
    )
    within_length = len(text.split()) <= 180

    score = 0
    score += 1 if has_intro else 0
    score += 1 if has_bullets else 0
    score += 1 if has_conclusion else 0
    score += 1 if mentions_metrics else 0
    score += 1 if within_length else 0

    return {
        "score": score,
        "has_intro": has_intro,
        "has_bullets": has_bullets,
        "has_conclusion": has_conclusion,
        "mentions_metrics": mentions_metrics,
        "within_length": within_length,
        "word_count": len(text.split()),
    }

# Calcul du score par reponse
scored = []
for row in all_results:
    metrics = score_answer(row.get("answer", ""))
    scored.append({
        "model": row.get("model"),
        "question": row.get("question"),
        **metrics,
    })

df_scores = pd.DataFrame(scored)
metric_cols = [
    "score",
    "has_intro",
    "has_bullets",
    "has_conclusion",
    "mentions_metrics",
    "within_length",
 ]
df_scores_summary = (
    df_scores.groupby("model")[metric_cols]
    .mean()
    .sort_values(by="score", ascending=False)
    .reset_index()
 )

df_scores_summary

,model,score,has_intro,has_bullets,has_conclusion,mentions_metrics,within_length
0,qwen2.5:7b,4.000000,0.0,1.0,1.000000,1.0,1.000000
1,deepseek-r1:7b,3.333333,0.0,1.0,0.666667,1.0,0.666667
2,llama3:8b,2.333333,0.0,0.0,1.000000,1.0,0.333333
